# Base-model selection pilot — password-locked sandbagging

This notebook implements `model_selection_spec.md` against the source-of-truth in `datasets_roster.md`. The primary selector uses **only** a fresh, quarantined row-1 generated batch: reverse complement, transcription, translation, GC content, ORF, and restriction sites. It performs no training and no free generation.

The frozen forced-choice body ends exactly at `Answer:` and contains no key line. Each model is scored by argmax over the next-token logits for contextual A/B/C/D. A 24-item LAB-Bench SeqQA slice is reported only as a contamination flag and never enters the decision accuracy.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

try:
    import pandas as pd
    from IPython.display import Markdown, display
except ImportError:
    pd = None
    Markdown = str
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'model_selection_pilot.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_selection_pilot import (
    CANDIDATES,
    CONTAMINATION_SLICE_SEED,
    DEFAULT_OUTPUT_DIR,
    LABBENCH_SLICE_COUNT,
    SELECTION_DATA_SEED,
    SELECTION_ITEM_COUNT,
    SELECTION_OPTION_SHUFFLE_SEED,
    apply_decision_rule,
    build_quarantined_batch,
    fetch_labbench_slice,
    labbench_rows_from_local,
    run_candidates,
    write_json,
)

OUTPUT_DIR = DEFAULT_OUTPUT_DIR
RUN_MODELS = os.getenv('RUN_MODEL_SELECTION', '0') == '1'
LOAD_IN_4BIT = os.getenv('MODEL_SELECTION_LOAD_IN_4BIT', '0') == '1'
BATCH_SIZE = int(os.getenv('MODEL_SELECTION_BATCH_SIZE', '4'))
LABBENCH_SEQQA_PATH = os.getenv('LABBENCH_SEQQA_PATH')
FETCH_LABBENCH_SLICE = os.getenv('FETCH_LABBENCH_SLICE', '1') == '1'

display({
    'run_models': RUN_MODELS,
    'load_in_4bit': LOAD_IN_4BIT,
    'batch_size': BATCH_SIZE,
    'output_dir': str(OUTPUT_DIR),
    'candidates': [candidate.model_id for candidate in CANDIDATES],
})

## 1. Fresh quarantined items and seeds

The generator seed and option-shuffle seed are checked against all recorded dataset-build seeds, including the generated dev/test offsets. The output lives under `data/quarantine/base_selection` and is marked `training_eligible: false`.

In [ ]:
generated_rows, batch_manifest = build_quarantined_batch(
    OUTPUT_DIR,
    count=SELECTION_ITEM_COUNT,
    data_seed=SELECTION_DATA_SEED,
    shuffle_seed=SELECTION_OPTION_SHUFFLE_SEED,
)
display(batch_manifest)

coverage = [
    {'task': key.split('|')[0], 'difficulty': key.split('|')[1], 'n': value}
    for key, value in batch_manifest['cell_counts'].items()
]
display(pd.DataFrame(coverage) if pd is not None else coverage)

sample_columns = ['pair_id', 'question', 'options', 'correct_letter', 'prompt']
samples = [{key: row[key] for key in sample_columns} for row in generated_rows[:6]]
display(pd.DataFrame(samples) if pd is not None else samples)

## 2. Small contamination-flag slice

This fixed 24-item LAB-Bench SeqQA slice is a glance only. Its score is excluded from generated accuracy, task survival, and size minimization. No PLSDB, MMLU-bio, PubMedQA, non-bio pool, CloningScenarios, or full held-out benchmark is loaded.

In [ ]:
labbench_rows = None
if RUN_MODELS:
    if LABBENCH_SEQQA_PATH:
        labbench_rows = labbench_rows_from_local(Path(LABBENCH_SEQQA_PATH), count=LABBENCH_SLICE_COUNT)
    elif FETCH_LABBENCH_SLICE:
        labbench_rows = fetch_labbench_slice(OUTPUT_DIR, count=LABBENCH_SLICE_COUNT)
    else:
        raise RuntimeError('A LAB-Bench SeqQA flag slice is required for a complete candidate report.')
    print({'slice': 'LAB-Bench SeqQA', 'role': 'contamination flag only', 'seed': CONTAMINATION_SLICE_SEED, 'n': len(labbench_rows)})
else:
    print('Model execution is off; the contamination slice is not fetched during the dry run.')

## 3. Sequential next-token scoring

Set `RUN_MODEL_SELECTION=1` before starting the kernel to execute. Models are loaded one at a time. For every candidate tokenizer, the notebook verifies that literal A/B/C/D are four distinct single tokens, then scores those token IDs at the exact frozen answer position. Full-string `prompt + letter` boundary stability is recorded as a diagnostic because BPE tokenizers may retokenize across `Answer:`; it is not the definition of a next token. A genuine multi-token failure still stops the pilot rather than changing the prompt or falling back to generation.

In [ ]:
summaries = []
if RUN_MODELS:
    summaries = run_candidates(
        generated_rows,
        output_dir=OUTPUT_DIR,
        labbench_rows=labbench_rows,
        candidates=CANDIDATES,
        batch_size=BATCH_SIZE,
        load_in_4bit=LOAD_IN_4BIT,
    )
else:
    print('Dry run complete. No model was loaded and no score was fabricated.')

## 4. Per-candidate metrics and flags

The task × difficulty table is primary. Aggregate generated accuracy, contextual answer-token checks, predicted-letter distribution, cost proxy, and the separate SeqQA contamination score are also shown.

In [ ]:
if summaries:
    primary_rows = []
    overview_rows = []
    token_rows = []
    bias_rows = []
    for summary in summaries:
        label = summary['candidate']['label']
        for cell, metrics in summary['accuracy_by_task_difficulty'].items():
            task, difficulty = cell.split('|')
            primary_rows.append({'candidate': label, 'task': task, 'difficulty': difficulty, **metrics})
        overview_rows.append({
            'candidate': label,
            'model_id': summary['candidate']['model_id'],
            'resolved_revision': summary['resolved_revision'],
            'params_B': summary['candidate']['parameters_billion'],
            'aggregate_generated_accuracy': summary['aggregate_verifiable_accuracy'],
            'LABBench_SeqQA_flag_accuracy': summary['contamination_flag_benchmark']['accuracy'],
            'LABBench_n': summary['contamination_flag_benchmark']['n'],
        })
        token_rows.append({'candidate': label, **summary['answer_token_check']['per_letter_single_token']})
        bias_rows.append({'candidate': label, **summary['predicted_letter_distribution'], 'TV_from_gold': summary['letter_distribution_total_variation_from_gold']})
    display(Markdown('### Primary: accuracy by task × difficulty'))
    display(pd.DataFrame(primary_rows) if pd is not None else primary_rows)
    display(Markdown('### Aggregate, revision, cost proxy, and contamination flag'))
    display(pd.DataFrame(overview_rows) if pd is not None else overview_rows)
    display(Markdown('### Contextual A/B/C/D single-token check'))
    display(pd.DataFrame(token_rows) if pd is not None else token_rows)
    display(Markdown('### Predicted-letter distribution (position-bias check)'))
    display(pd.DataFrame(bias_rows) if pd is not None else bias_rows)
else:
    print('Metrics will appear after RUN_MODEL_SELECTION=1 completes.')

## 5. Filter, then minimize

A task clears only when both difficulty tiers are at least 70% and the task is not saturated in both tiers. Candidate-level filters reject fewer than two cleared task types, heavy letter bias, suspiciously near-perfect SeqQA, or aggregate saturation. Among survivors, nominal parameter count is minimized. Tasks cleared by no candidate are explicitly cut; if no task clears, the result requests a larger model class.

In [ ]:
decision = None
if summaries:
    decision = apply_decision_rule(summaries)
    write_json(OUTPUT_DIR / 'decision.json', decision)
    decision_rows = [{
        'candidate': row['label'],
        'revision': row['resolved_revision'],
        'params_B': row['parameters_billion'],
        'cleared_tasks': ', '.join(row['cleared_tasks']),
        'letter_bias': row['heavy_letter_bias'],
        'contamination': row['contamination_flag'],
        'saturated': row['aggregate_saturated'],
        'survives': row['survives_filter'],
        'drop_reasons': '; '.join(row['drop_reasons']),
    } for row in decision['candidates']]
    display(pd.DataFrame(decision_rows) if pd is not None else decision_rows)
    display(Markdown(
        f"**Chosen base + revision:** `{decision['chosen_base']}`\n\n"
        f"**Surviving tasks:** {decision['surviving_tasks']}\n\n"
        f"**Tasks to cut:** {decision['tasks_to_cut_from_spine']}\n\n"
        f"**Rationale:** {decision['rationale']}"
    ))
else:
    print('Decision intentionally pending: run all candidates and the contamination-flag slice first.')